# Example 1.2.4 / 3: Sparse Matrices in amplpy

Companion notebook for **1.2.4 Sparse Parameters** in the AMPL lab Documentation, section "Sparse Matrices in amplpy". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Sparse Matrices in amplpy

A multidimensional AMPL parameter maps naturally to a Python dictionary whose keys are tuples.
For a three-dimensional parameter, use three-element keys such as `(product, plant, week)`.
Assign the corresponding tuple set from `data.keys()` when the parameter exists only at those
keys:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
set PLANT;
set WEEK;
set VALID within {PROD, PLANT, WEEK};
param shipping_cost {VALID};

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.set["PLANT"] = ["GARY", "PITT"]
ampl.set["WEEK"] = [1, 2]
shipping_cost = {
    ("bands", "GARY", 1): 14,
    ("bands", "PITT", 1): 16,
    ("coils", "GARY", 2): 11,
}

ampl.set["VALID"] = list(shipping_cost)
ampl.param["shipping_cost"] = shipping_cost

In [5]:
# run.py
print(ampl.param["shipping_cost"].to_dict())

{('bands', 'GARY', 1): 14, ('bands', 'PITT', 1): 16, ('coils', 'GARY', 2): 11}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils;
set PLANT := GARY PITT;
set WEEK := 1 2;
set VALID := (bands,GARY,1) (bands,PITT,1) (coils,GARY,2);
param shipping_cost :=
  bands GARY 1  14
  bands PITT 1  16
  coils GARY 2  11 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display shipping_cost;

shipping_cost :=
bands GARY 1   14
bands PITT 1   16
coils GARY 2   11
;

